In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)  

In [ ]:
n_days = 40
dates = pd.date_range("2026-10-01", periods=n_days, freq="D")

users = np.random.randint(500, 1500, n_days)
sessions = (users * np.random.uniform(1.2, 1.8, n_days)).astype(int)
revenue = (sessions * np.random.uniform(0.8, 1.4, n_days)).round(2)

df = pd.DataFrame({
    "date": dates,
    "users": users,
    "sessions": sessions,
    "revenue": revenue,
})
print(df.head())

corr = df[["users", "sessions", "revenue"]].corr()
print("\nКореляційна матриця:")
print(corr)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
pairs = [("users", "sessions"), ("users", "revenue"), ("sessions", "revenue")]
for ax, (x, y) in zip(axs, pairs):
    ax.scatter(df[x], df[y], alpha=0.7)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f"{x} vs {y}")
    ax.grid(True)
fig.suptitle("Діаграми розсіювання")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(df["date"], df["revenue"], marker="o")
plt.xlabel("Дата")
plt.ylabel("Виручка")
plt.title("Виручка за датами")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
n = 500
group_a = pd.DataFrame({"group": "A", "converted": np.random.binomial(1, 0.10, n)})
group_b = pd.DataFrame({"group": "B", "converted": np.random.binomial(1, 0.13, n)})
ab = pd.concat([group_a, group_b], ignore_index=True)

stats = ab.groupby("group")["converted"].agg(["sum", "count", "mean"])
stats.columns = ["conversions", "n", "rate"]

# 95% довірчий інтервал (нормальне наближення)
z = 1.96
stats["se"] = np.sqrt(stats["rate"] * (1 - stats["rate"]) / stats["n"])
stats["ci_low"] = (stats["rate"] - z * stats["se"]).clip(lower=0)
stats["ci_high"] = stats["rate"] + z * stats["se"]
print(stats.round(4))

rate_a, rate_b = stats.loc["A", "rate"], stats.loc["B", "rate"]
print(f"\nКонверсія A: {rate_a:.2%}")
print(f"Конверсія B: {rate_b:.2%}")
print(f"Абсолютна різниця: {rate_b - rate_a:.2%}")
print(f"Відносна зміна: {(rate_b - rate_a) / rate_a:.2%}")

err = [stats["rate"] - stats["ci_low"], stats["ci_high"] - stats["rate"]]
plt.figure(figsize=(6, 5))
plt.bar(stats.index, stats["rate"], yerr=err, capsize=8,
        color=["tab:blue", "tab:orange"])
plt.xlabel("Група")
plt.ylabel("Конверсія")
plt.title("Конверсія за групами з 95% ДІ")
plt.show()

In [ ]:
population = np.random.exponential(scale=2.0, size=50_000)
print(f"Генеральна сукупність: середнє={population.mean():.3f}, std={population.std():.3f}")

n_samples = 2000
sizes = [5, 50]

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, n in zip(axs, sizes):
    sample_means = np.array([
        np.random.choice(population, size=n).mean()
        for _ in range(n_samples)
    ])
    ax.hist(sample_means, bins=40, edgecolor="black", alpha=0.8)
    ax.set_title(f"Вибіркові середні, n = {n}")
    ax.set_xlabel("Вибіркове середнє")
    ax.set_ylabel("Частота")
    print(f"n={n}: середнє={sample_means.mean():.3f}, std={sample_means.std():.3f}")

plt.tight_layout()
plt.show()

In [ ]:
days = 90
dates = pd.date_range("2026-08-01", periods=days, freq="D")
trend = np.linspace(100, 150, days)
seasonal = 10 * np.sin(np.arange(days) * 2 * np.pi / 7)
noise = np.random.normal(0, 8, days)

sales_df = pd.DataFrame({"date": dates, "sales": (trend + seasonal + noise).round(2)})

window = 7
sales_df["rolling_mean"] = sales_df["sales"].rolling(window).mean()
sales_df["rolling_std"] = sales_df["sales"].rolling(window).std()

print(sales_df.head(10))

plt.figure(figsize=(11, 5))
plt.plot(sales_df["date"], sales_df["sales"], label="Продажі", alpha=0.6)
plt.plot(sales_df["date"], sales_df["rolling_mean"], label=f"Ковзне середнє ({window} дн.)", linewidth=2)
plt.xlabel("Дата")
plt.ylabel("Продажі")
plt.title("Продажі та ковзне середнє")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 4))
plt.plot(sales_df["date"], sales_df["rolling_std"], color="tab:red")
plt.xlabel("Дата")
plt.ylabel("Ковзне стандартне відхилення")
plt.title(f"Ковзне стандартне відхилення (вікно {window} дн.)")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()